In [ ]:

import cv2

import numpy as np

import glob

import os

import base64

import json

from ultralytics import YOLO

from google.colab import files

from IPython.display import HTML, display



# --- 1. 배경 이미지 로드 ---

CLEAN_BREADBOARD_IMG = '/content/clean_breadboard.png'

cb_canvas = cv2.imread(CLEAN_BREADBOARD_IMG)



if cb_canvas is None:

    print("🚨 [오류] 'clean_breadboard.png' 파일을 찾을 수 없습니다. 왼쪽 폴더에 업로드 해주세요.")

else:

    CB_HEIGHT, CB_WIDTH, _ = cb_canvas.shape

    roi_x1, roi_x2 = int(CB_WIDTH * 0.10), int(CB_WIDTH * 0.90)

    roi_y1, roi_y2 = int(CB_HEIGHT * 0.15), int(CB_HEIGHT * 0.85)

    roi_w, roi_h = roi_x2 - roi_x1, roi_y2 - roi_y1



# --- 2. 빵판 구멍 자동 인식 (글씨 및 기호 완벽 차단) ---
    gray = cv2.cvtColor(cb_canvas, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

    temp_holes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        bbox_area = w * h
        contour_area = cv2.contourArea(cnt) # 실제 색칠된 픽셀의 면적

        if h == 0 or w == 0 or contour_area == 0: continue

        aspect_ratio = w / float(h)
        extent = contour_area / float(bbox_area) # 박스 대비 실제 면적 비율 (원은 약 0.78)

        # 1차 필터: 비율이 정사각형에 가깝고(0.75~1.3),
        # 내부가 꽉 차 있는(extent > 0.65) 형태만 통과 ('+' 기호 등은 여기서 탈락)
        if 20 < bbox_area < 1000 and 0.75 < aspect_ratio < 1.3 and extent > 0.65:
            temp_holes.append({"x": x + w//2, "y": y + h//2, "area": bbox_area})

    holes_data = []
    if not temp_holes:
        print("🚨 [오류] 그림 파일에서 구멍을 전혀 찾지 못했습니다!")
    else:
        # 2차 절대 필터: 찾아낸 구멍들의 '중간값(가장 흔한 크기)' 도출
        median_area = np.median([h['area'] for h in temp_holes])

        for h in temp_holes:
            # 빵판 구멍은 크기가 일정함! 중간값 기준 위아래 30% 오차를 벗어나면 글씨/먼지로 간주하고 탈락
            if median_area * 0.7 < h['area'] < median_area * 1.3:
                holes_data.append({"x": h['x'], "y": h['y'], "net_id": None})



        # 구역별 맞춤형 연결 로직 (이 아래는 기존 코드와 동일합니다)

        for i in range(len(holes_data)):

            for j in range(i+1, len(holes_data)):

                h1, h2 = holes_data[i], holes_data[j]

                dx, dy = abs(h1['x'] - h2['x']), abs(h1['y'] - h2['y'])



                # 1번 구멍의 구역 찾기

                if h1['x'] < Z1_END: z1 = 1

                elif h1['x'] < Z2_END: z1 = 2

                elif h1['x'] > Z4_START: z1 = 4

                elif h1['x'] >= Z3_START: z1 = 3

                else: z1 = 0



                # 2번 구멍의 구역 찾기

                if h2['x'] < Z1_END: z2 = 1

                elif h2['x'] < Z2_END: z2 = 2

                elif h2['x'] > Z4_START: z2 = 4

                elif h2['x'] >= Z3_START: z2 = 3

                else: z2 = 0



                # 같은 구역 안에 있는 구멍들만 묶습니다!

                if z1 == z2 and z1 != 0:

                    if z1 == 2 or z1 == 3: # [메인 구역] 가로로 인접하면 묶음 (Y가 같아야 함)

                        if dy < TOL: union(i, j)

                    elif z1 == 1 or z1 == 4: # [전원 구역] 세로로 인접하면 묶음 (X가 같아야 함)

                        if dx < TOL: union(i, j)



        # Net 이름 부여

        net_dict = {}

        net_counter = 1

        for i in range(len(holes_data)):

            root = find(i)

            if root not in net_dict:

                net_dict[root] = f"NET_{net_counter:03d}"

                net_counter += 1

            holes_data[i]['net_id'] = net_dict[root]



        print(f"🎯 인식된 구멍: {len(holes_data)}개 / 생성된 회로 라인(Net): {net_counter-1}개 (성공!)")



        # --- 4. 모델 추론 ---

        weight_paths = glob.glob('/content/runs/detect/*/weights/best.pt')

        if not weight_paths:

            print("🚨 [오류] 학습된 모델을 찾을 수 없습니다.")

        else:

            trained_model = YOLO(max(weight_paths, key=os.path.getctime))



            print("\n👇 [파일 선택] 버튼을 눌러 실제 빵판 사진을 업로드해주세요!")

            uploaded = files.upload()



            if uploaded:

                real_image_path = list(uploaded.keys())[0]

                real_img = cv2.imread(real_image_path)

                real_h, real_w, _ = real_img.shape



                results = trained_model(real_image_path, conf=0.5)

                result = results[0]



                components_data = []

                for i, box in enumerate(result.boxes):

                    class_name = trained_model.names[int(box.cls[0].item())].lower()

                    xmin, ymin, xmax, ymax = map(int, box.xyxy[0].tolist())



                    rel_xmin, rel_ymin = xmin / real_w, ymin / real_h

                    rel_xmax, rel_ymax = xmax / real_w, ymax / real_h



                    cb_xmin = roi_x1 + int(rel_xmin * roi_w)

                    cb_ymin = roi_y1 + int(rel_ymin * roi_h)

                    cb_w = int((rel_xmax - rel_xmin) * roi_w)

                    cb_h = int((rel_ymax - rel_ymin) * roi_h)



                    components_data.append({

                        "id": i, "name": f"{class_name}_{i+1}", "type": class_name,

                        "x": cb_xmin, "y": cb_ymin, "w": cb_w, "h": cb_h

                    })



                with open(CLEAN_BREADBOARD_IMG, "rb") as image_file:

                    encoded_string = base64.b64encode(image_file.read()).decode()

                bg_base64 = f"data:image/png;base64,{encoded_string}"



                json_components = json.dumps(components_data)

                json_holes = json.dumps(holes_data)



                # --- 5. HTML+JS 시뮬레이터 (선 그리기 로직 개선) ---

                html_code = f"""

                <div style="text-align: center; font-family: sans-serif;">

                    <h3 style="color: #333;">💡 이제 <span style="color: #00AA00; font-weight:bold;">초록색 연결망</span>이 완벽하게 보일 겁니다!</h3>

                    <p>같은 초록색 선 위에 부품들의 다리를 꽂고 추출 버튼을 눌러보세요.</p>

                    <div style="display: inline-block; position: relative;">

                        <canvas id="breadboardCanvas" width="{CB_WIDTH}" height="{CB_HEIGHT}"

                                style="border: 2px solid #555; max-width: 100%; height: auto; box-shadow: 2px 2px 10px rgba(0,0,0,0.3);"></canvas>

                    </div>

                    <br>

                    <button onclick="generateNetlist()" style="padding: 12px 24px; font-size: 16px; font-weight: bold; margin-top: 15px; cursor: pointer; background-color: #007BFF; color: white; border: none; border-radius: 5px;">✅ 넷리스트 추출하기</button>

                    <pre id="outputConsole" style="text-align: left; background: #222; color: #0f0; padding: 15px; margin-top: 15px; max-height: 400px; overflow-y: auto; border-radius: 5px; font-size: 15px; width: 80%; margin-left: auto; margin-right: auto; line-height: 1.6;"></pre>

                </div>



                <script>

                    const canvas = document.getElementById('breadboardCanvas');

                    const ctx = canvas.getContext('2d');

                    const outputConsole = document.getElementById('outputConsole');



                    let items = {json_components};

                    let holes = {json_holes};

                    let bgImg = new Image();

                    bgImg.src = "{bg_base64}";



                    let isDragging = false, isResizing = false;

                    let dragTarget = null, offsetX, offsetY;

                    const HANDLE_SIZE = 12;



                    let netGroups = {{}};

                    holes.forEach(h => {{

                        if (!netGroups[h.net_id]) netGroups[h.net_id] = [];

                        netGroups[h.net_id].push(h);

                    }});



                    bgImg.onload = () => draw();



                    function getNearestHole(px, py) {{

                        let minDist = Infinity, nearest = null;

                        holes.forEach(h => {{

                            let d = Math.hypot(h.x - px, h.y - py);

                            if (d < minDist) {{ minDist = d; nearest = h; }}

                        }});

                        return nearest;

                    }}



                    function draw() {{

                        ctx.clearRect(0, 0, canvas.width, canvas.height);

                        ctx.drawImage(bgImg, 0, 0, canvas.width, canvas.height);



                        // 🌟 빵판 내부 연결 실선 그리기 (그리기 로직 개선)

                        ctx.lineWidth = 6; // 선을 더 두껍게

                        ctx.strokeStyle = "rgba(0, 220, 50, 0.6)"; // 더 밝은 초록색

                        ctx.lineCap = "round";

                        ctx.lineJoin = "round";



                        for (let net in netGroups) {{

                            let hList = netGroups[net];

                            if (hList.length > 1) {{

                                // 세로줄인지 가로줄인지 파악하여 정렬 방향 결정

                                let dx = Math.abs(hList[0].x - hList[hList.length-1].x);

                                let dy = Math.abs(hList[0].y - hList[hList.length-1].y);



                                if (dx > dy) hList.sort((a, b) => a.x - b.x); // 가로 정렬

                                else hList.sort((a, b) => a.y - b.y); // 세로 정렬



                                ctx.beginPath();

                                ctx.moveTo(hList[0].x, hList[0].y);

                                for(let i=1; i<hList.length; i++) {{

                                    ctx.lineTo(hList[i].x, hList[i].y);

                                }}

                                ctx.stroke();

                            }}

                        }}



                        // 부품 그리기

                        items.forEach(item => {{

                            ctx.fillStyle = "rgba(255, 0, 255, 0.15)";

                            ctx.strokeStyle = "rgba(255, 0, 255, 1)";

                            ctx.lineWidth = 2;

                            ctx.fillRect(item.x, item.y, item.w, item.h);

                            ctx.strokeRect(item.x, item.y, item.w, item.h);



                            ctx.fillStyle = "#8B008B";

                            ctx.font = "bold 16px Arial";

                            ctx.fillText(item.name, item.x, item.y - 8);



                            ctx.fillStyle = "red";

                            ctx.fillRect(item.x + item.w - HANDLE_SIZE, item.y + item.h - HANDLE_SIZE, HANDLE_SIZE, HANDLE_SIZE);



                            drawPins(item);

                        }});

                    }}



                    function drawPins(item) {{

                        let pins = getPinCoords(item);

                        pins.forEach(pin => {{

                            let nearest = getNearestHole(pin.x, pin.y);

                            if (nearest) {{

                                ctx.beginPath();

                                ctx.moveTo(pin.x, pin.y); ctx.lineTo(nearest.x, nearest.y);

                                ctx.strokeStyle = "rgba(0,0,0,0.5)"; ctx.setLineDash([5, 5]); ctx.stroke(); ctx.setLineDash([]);



                                ctx.beginPath();

                                ctx.arc(nearest.x, nearest.y, 7, 0, 2 * Math.PI, false);

                                ctx.strokeStyle = "black"; ctx.lineWidth = 2; ctx.stroke();

                            }}



                            ctx.beginPath();

                            ctx.arc(pin.x, pin.y, 6, 0, 2 * Math.PI, false);

                            ctx.fillStyle = pin.color; ctx.fill(); ctx.strokeStyle = 'white'; ctx.stroke();

                        }});

                    }}



                    function getPinCoords(item) {{

                        let pins = [];

                        if (['fet', 'mosfet'].includes(item.type)) {{

                            let mid_x = item.x + (item.w / 2);

                            pins.push({{name: "Source", x: mid_x, y: item.y, color: "blue"}});

                            pins.push({{name: "Drain", x: mid_x, y: item.y + (item.h / 2), color: "green"}});

                            pins.push({{name: "Gate", x: mid_x, y: item.y + item.h, color: "red"}});

                        }} else if (['res', 'led', 'cap', 'jump'].includes(item.type)) {{

                            if (item.w > item.h) {{

                                let mid_y = item.y + (item.h / 2);

                                pins.push({{name: "Pin1", x: item.x, y: mid_y, color: "red"}});

                                pins.push({{name: "Pin2", x: item.x + item.w, y: mid_y, color: "red"}});

                            }} else {{

                                let mid_x = item.x + (item.w / 2);

                                pins.push({{name: "Pin1", x: mid_x, y: item.y, color: "red"}});

                                pins.push({{name: "Pin2", x: mid_x, y: item.y + item.h, color: "red"}});

                            }}

                        }} else {{

                            pins.push({{name: "Pin1", x: item.x, y: item.y + item.h, color: "red"}});

                            pins.push({{name: "Pin2", x: item.x + item.w, y: item.y + item.h, color: "red"}});

                        }}

                        return pins;

                    }}



                    function getMousePos(evt) {{

                        const rect = canvas.getBoundingClientRect();

                        const scaleX = canvas.width / rect.width; const scaleY = canvas.height / rect.height;

                        return {{ x: (evt.clientX - rect.left) * scaleX, y: (evt.clientY - rect.top) * scaleY }};

                    }}



                    canvas.addEventListener('mousedown', function(e) {{

                        const pos = getMousePos(e);

                        for (let i = items.length - 1; i >= 0; i--) {{

                            let item = items[i];

                            if (pos.x >= item.x + item.w - HANDLE_SIZE && pos.x <= item.x + item.w && pos.y >= item.y + item.h - HANDLE_SIZE && pos.y <= item.y + item.h) {{

                                isResizing = true; dragTarget = item; canvas.style.cursor = 'nwse-resize'; return;

                            }}

                            if (pos.x >= item.x && pos.x <= item.x + item.w && pos.y >= item.y && pos.y <= item.y + item.h) {{

                                isDragging = true; dragTarget = item; offsetX = pos.x - item.x; offsetY = pos.y - item.y; canvas.style.cursor = 'grabbing'; return;

                            }}

                        }}

                    }});



                    canvas.addEventListener('mousemove', function(e) {{

                        const pos = getMousePos(e);

                        if (isDragging && dragTarget) {{ dragTarget.x = pos.x - offsetX; dragTarget.y = pos.y - offsetY; draw(); }}

                        else if (isResizing && dragTarget) {{

                            let newW = pos.x - dragTarget.x; let newH = pos.y - dragTarget.y;

                            if (newW > 20) dragTarget.w = newW; if (newH > 20) dragTarget.h = newH;

                            draw();

                        }}

                    }});



                    canvas.addEventListener('mouseup', function(e) {{

                        isDragging = false; isResizing = false; dragTarget = null; canvas.style.cursor = 'default'; draw();

                    }});



                    canvas.addEventListener('mouseleave', function(e) {{

                        isDragging = false; isResizing = false; dragTarget = null; canvas.style.cursor = 'default'; draw();

                    }});



                    function generateNetlist() {{

                        let netlistMap = {{}};



                        items.forEach(item => {{

                            let pins = getPinCoords(item);

                            pins.forEach(pin => {{

                                let nearestHole = getNearestHole(pin.x, pin.y);

                                if (nearestHole && nearestHole.net_id) {{

                                    if (!netlistMap[nearestHole.net_id]) netlistMap[nearestHole.net_id] = [];

                                    netlistMap[nearestHole.net_id].push(`${{item.name}}(${{pin.name}})`);

                                }}

                            }});

                        }});



                        let out = "🔌 [최종 추출된 전기적 회로 넷리스트]\\n\\n";

                        let validConnections = false;



                        out += "🔗 [상호 연결된 노드]\\n";

                        for (let net in netlistMap) {{

                            if (netlistMap[net].length > 1) {{

                                out += `  - ${{net}} :  ${{netlistMap[net].join("  ↔  ")}}\\n`;

                                validConnections = true;

                            }}

                        }}



                        if (!validConnections) out += "  (아직 연결된 부품이 없습니다. 초록색 선을 공유하도록 다리를 놔주세요!)\\n";



                        out += "\\n📌 [단일 연결 (혼자 꽂힌 핀)]\\n";

                        for (let net in netlistMap) {{

                            if (netlistMap[net].length === 1) {{

                                out += `  - ${{net}} :  ${{netlistMap[net][0]}}\\n`;

                            }}

                        }}



                        outputConsole.innerText = out;

                    }}

                </script>

                """



                display(HTML(html_code))

import cv2
import numpy as np
import glob
import os
import base64
import json
from ultralytics import YOLO
from google.colab import files
from IPython.display import HTML, display

# --- 1. 배경 이미지 로드 ---
CLEAN_BREADBOARD_IMG = '/content/clean_breadboard.png'
cb_canvas = cv2.imread(CLEAN_BREADBOARD_IMG)

if cb_canvas is None:
    print("🚨 [오류] 'clean_breadboard.png' 파일을 찾을 수 없습니다. 왼쪽 폴더에 업로드 해주세요.")
else:
    CB_HEIGHT, CB_WIDTH, _ = cb_canvas.shape
    roi_x1, roi_x2 = int(CB_WIDTH * 0.10), int(CB_WIDTH * 0.90)
    roi_y1, roi_y2 = int(CB_HEIGHT * 0.15), int(CB_HEIGHT * 0.85)
    roi_w, roi_h = roi_x2 - roi_x1, roi_y2 - roi_y1

    # --- 2. 🌟 빵판 구멍 자동 인식 (글씨 및 기호 완벽 차단) ---
    gray = cv2.cvtColor(cb_canvas, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

    temp_holes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        bbox_area = w * h
        contour_area = cv2.contourArea(cnt)

        if h == 0 or w == 0 or contour_area == 0: continue

        aspect_ratio = w / float(h)
        extent = contour_area / float(bbox_area)

        # 1차 필터: 모양이 둥글고 꽉 찬 형태만 통과 ('+', 'a' 기호 등 1차 탈락)
        if 20 < bbox_area < 1000 and 0.75 < aspect_ratio < 1.3 and extent > 0.65:
            temp_holes.append({"x": x + w//2, "y": y + h//2, "area": bbox_area})

    holes_data = []
    if not temp_holes:
        print("🚨 [오류] 그림 파일에서 구멍을 전혀 찾지 못했습니다!")
    else:
        # 2차 절대 필터: 구멍들의 '중간값' 크기 도출 (크기가 다른 찌꺼기 완벽 차단)
        median_area = np.median([h['area'] for h in temp_holes])

        for h in temp_holes:
            if median_area * 0.7 < h['area'] < median_area * 1.3:
                holes_data.append({"x": h['x'], "y": h['y'], "net_id": None})

        # --- 3. 🌟 4구역(Zone) 기반 Net 그룹화 및 전원선 구분 ---
        x_coords = [h['x'] for h in holes_data]
        min_x, max_x = min(x_coords), max(x_coords)
        span_x = max_x - min_x

        # a열이 전원부로 끌려가지 않도록 구역 범위를 정밀하게 축소
        Z1_END = min_x + span_x * 0.12    # 왼쪽 전원부 (+, -)
        Z2_END = min_x + span_x * 0.48    # 왼쪽 메인부 (a~e)
        Z3_START = min_x + span_x * 0.52  # 오른쪽 메인부 (f~j)
        Z4_START = min_x + span_x * 0.88  # 오른쪽 전원부 (+, -)

        TOL = span_x * 0.015

        net_map = {i: i for i in range(len(holes_data))}
        def find(i):
            if net_map[i] == i: return i
            net_map[i] = find(net_map[i])
            return net_map[i]
        def union(i, j):
            root_i, root_j = find(i), find(j)
            if root_i != root_j: net_map[root_i] = root_j

        for i in range(len(holes_data)):
            for j in range(i+1, len(holes_data)):
                h1, h2 = holes_data[i], holes_data[j]
                dx, dy = abs(h1['x'] - h2['x']), abs(h1['y'] - h2['y'])

                if h1['x'] < Z1_END: z1 = 1
                elif h1['x'] < Z2_END: z1 = 2
                elif h1['x'] > Z4_START: z1 = 4
                elif h1['x'] >= Z3_START: z1 = 3
                else: z1 = 0

                if h2['x'] < Z1_END: z2 = 1
                elif h2['x'] < Z2_END: z2 = 2
                elif h2['x'] > Z4_START: z2 = 4
                elif h2['x'] >= Z3_START: z2 = 3
                else: z2 = 0

                if z1 == z2 and z1 != 0:
                    if z1 == 2 or z1 == 3: # 가로 묶음
                        if dy < TOL: union(i, j)
                    elif z1 == 1 or z1 == 4: # 세로 묶음
                        if dx < TOL: union(i, j)

        net_groups = {}
        for i in range(len(holes_data)):
            root = find(i)
            if root not in net_groups:
                net_groups[root] = []
            net_groups[root].append(holes_data[i])

        net_dict = {}
        net_counter = 1

        z1_roots, z4_roots = [], []

        for root, holes in net_groups.items():
            avg_x = sum(h['x'] for h in holes) / len(holes)
            if len(holes) < 3:
                net_dict[root] = f"IGNORE_{root}"
                continue

            if avg_x < Z1_END: z1_roots.append((root, avg_x))
            elif avg_x > Z4_START: z4_roots.append((root, avg_x))
            else:
                net_dict[root] = f"NET_{net_counter:03d}"
                net_counter += 1

        # 전원선 + , - 라벨링
        z1_roots.sort(key=lambda item: item[1])
        if len(z1_roots) >= 2:
            net_dict[z1_roots[0][0]] = "LEFT_PWR_+"
            net_dict[z1_roots[-1][0]] = "LEFT_PWR_-"

        z4_roots.sort(key=lambda item: item[1])
        if len(z4_roots) >= 2:
            net_dict[z4_roots[0][0]] = "RIGHT_PWR_+"
            net_dict[z4_roots[-1][0]] = "RIGHT_PWR_-"

        valid_nets = set()
        for i in range(len(holes_data)):
            root = find(i)
            net_name = net_dict.get(root, f"NET_UNK_{root}")
            holes_data[i]['net_id'] = net_name
            if "IGNORE" not in net_name:
                valid_nets.add(net_name)

        print(f"🎯 인식된 구멍: {len(holes_data)}개 / 생성된 회로 라인(Net): {len(valid_nets)}개 (성공!)")

        # --- 4. 모델 추론 ---
        weight_paths = glob.glob('/content/runs/detect/*/weights/best.pt')
        if not weight_paths:
            print("🚨 [오류] 학습된 모델을 찾을 수 없습니다. (모델 경로를 확인하세요)")
        else:
            trained_model = YOLO(max(weight_paths, key=os.path.getctime))

            print("\n👇 [파일 선택] 버튼을 눌러 실제 빵판 사진을 업로드해주세요!")
            uploaded = files.upload()

            if uploaded:
                real_image_path = list(uploaded.keys())[0]
                real_img = cv2.imread(real_image_path)
                real_h, real_w, _ = real_img.shape

                results = trained_model(real_image_path)
                result = results[0]

                components_data = []
                for i, box in enumerate(result.boxes):
                    class_name = trained_model.names[int(box.cls[0].item())].lower()
                    xmin, ymin, xmax, ymax = map(int, box.xyxy[0].tolist())

                    rel_xmin, rel_ymin = xmin / real_w, ymin / real_h
                    rel_xmax, rel_ymax = xmax / real_w, ymax / real_h

                    cb_xmin = roi_x1 + int(rel_xmin * roi_w)
                    cb_ymin = roi_y1 + int(rel_ymin * roi_h)
                    cb_w = int((rel_xmax - rel_xmin) * roi_w)
                    cb_h = int((rel_ymax - rel_ymin) * roi_h)

                    components_data.append({
                        "id": i, "name": f"{class_name}_{i+1}", "type": class_name,
                        "x": cb_xmin, "y": cb_ymin, "w": cb_w, "h": cb_h
                    })

                with open(CLEAN_BREADBOARD_IMG, "rb") as image_file:
                    encoded_string = base64.b64encode(image_file.read()).decode()
                bg_base64 = f"data:image/png;base64,{encoded_string}"

                json_components = json.dumps(components_data)
                json_holes = json.dumps(holes_data)

                # --- 5. HTML+JS 시뮬레이터 ---
                html_code = f"""
                <div style="text-align: center; font-family: sans-serif;">
                    <h3 style="color: #333;">💡 소자 위치가 어긋났나요? 슬라이더로 빵판에 싹 맞추세요!</h3>

                    <div style="background: #eef; padding: 15px; border-radius: 8px; display: inline-block; margin-bottom: 15px; text-align: left; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);">
                        <strong style="display:block; margin-bottom: 8px; color:#222;">🛠️ 전체 소자 일괄 조절 (대략적인 위치를 맞춘 후, 개별 드래그 하세요)</strong>
                        <label style="display:inline-block; width: 48%;">↔ 가로 이동: <input type="range" id="shiftX" min="-500" max="500" value="0" oninput="applyGlobalTransform()"></label>
                        <label style="display:inline-block; width: 48%;">↕ 세로 이동: <input type="range" id="shiftY" min="-500" max="500" value="0" oninput="applyGlobalTransform()"></label><br>
                        <label style="display:inline-block; width: 48%;">🔍 전체 크기: <input type="range" id="scaleAll" min="0.3" max="3.0" step="0.05" value="1.0" oninput="applyGlobalTransform()"></label>
                        <label style="display:inline-block; width: 48%;">↔ 가로 폭만: <input type="range" id="scaleX" min="0.5" max="2.0" step="0.05" value="1.0" oninput="applyGlobalTransform()"></label>
                    </div>

                    <div style="display: inline-block; position: relative;">
                        <canvas id="breadboardCanvas" width="{CB_WIDTH}" height="{CB_HEIGHT}"
                                style="border: 2px solid #555; max-width: 100%; height: auto; box-shadow: 2px 2px 10px rgba(0,0,0,0.3);"></canvas>
                    </div>
                    <br>
                    <button onclick="generateNetlist()" style="padding: 12px 24px; font-size: 16px; font-weight: bold; margin-top: 15px; cursor: pointer; background-color: #007BFF; color: white; border: none; border-radius: 5px;">✅ 넷리스트 추출하기</button>
                    <pre id="outputConsole" style="text-align: left; background: #222; color: #0f0; padding: 15px; margin-top: 15px; max-height: 400px; overflow-y: auto; border-radius: 5px; font-size: 15px; width: 80%; margin-left: auto; margin-right: auto; line-height: 1.6;"></pre>
                </div>

                <script>
                    const canvas = document.getElementById('breadboardCanvas');
                    const ctx = canvas.getContext('2d');
                    const outputConsole = document.getElementById('outputConsole');

                    let items = {json_components};
                    let originalItems = JSON.parse(JSON.stringify(items));

                    let holes = {json_holes};
                    let bgImg = new Image();
                    bgImg.src = "{bg_base64}";

                    let isDragging = false, isResizing = false;
                    let dragTarget = null, offsetX, offsetY;
                    const HANDLE_SIZE = 12;

                    let netGroups = {{}};
                    holes.forEach(h => {{
                        if (!netGroups[h.net_id]) netGroups[h.net_id] = [];
                        netGroups[h.net_id].push(h);
                    }});

                    bgImg.onload = () => draw();

                    function applyGlobalTransform() {{
                        let sx = parseInt(document.getElementById('shiftX').value);
                        let sy = parseInt(document.getElementById('shiftY').value);
                        let scale = parseFloat(document.getElementById('scaleAll').value);
                        let scaleX = parseFloat(document.getElementById('scaleX').value);

                        let cx = canvas.width / 2;
                        let cy = canvas.height / 2;

                        for (let i = 0; i < items.length; i++) {{
                            items[i].w = originalItems[i].w * scale * scaleX;
                            items[i].h = originalItems[i].h * scale;
                            items[i].x = (originalItems[i].x - cx) * scale * scaleX + cx + sx;
                            items[i].y = (originalItems[i].y - cy) * scale + cy + sy;
                        }}
                        draw();
                    }}

                    function getNearestHole(px, py) {{
                        let minDist = Infinity, nearest = null;
                        holes.forEach(h => {{
                            let d = Math.hypot(h.x - px, h.y - py);
                            if (d < minDist) {{ minDist = d; nearest = h; }}
                        }});
                        return nearest;
                    }}

                    function draw() {{
                        ctx.clearRect(0, 0, canvas.width, canvas.height);
                        ctx.drawImage(bgImg, 0, 0, canvas.width, canvas.height);

                        ctx.lineWidth = 6;
                        ctx.lineCap = "round";
                        ctx.lineJoin = "round";

                        for (let net in netGroups) {{
                            if (net.includes("IGNORE")) continue;

                            let hList = netGroups[net];
                            if (hList.length > 1) {{
                                let dx = Math.abs(hList[0].x - hList[hList.length-1].x);
                                let dy = Math.abs(hList[0].y - hList[hList.length-1].y);

                                if (dx > dy) hList.sort((a, b) => a.x - b.x);
                                else hList.sort((a, b) => a.y - b.y);

                                if (net.includes("+")) {{
                                    ctx.strokeStyle = "rgba(255, 50, 50, 0.6)";
                                }} else if (net.includes("-")) {{
                                    ctx.strokeStyle = "rgba(50, 50, 255, 0.6)";
                                }} else {{
                                    ctx.strokeStyle = "rgba(0, 220, 50, 0.6)";
                                }}

                                ctx.beginPath();
                                ctx.moveTo(hList[0].x, hList[0].y);
                                for(let i=1; i<hList.length; i++) {{
                                    ctx.lineTo(hList[i].x, hList[i].y);
                                }}
                                ctx.stroke();
                            }}
                        }}

                        items.forEach(item => {{
                            ctx.fillStyle = "rgba(255, 0, 255, 0.15)";
                            ctx.strokeStyle = "rgba(255, 0, 255, 1)";
                            ctx.lineWidth = 2;
                            ctx.fillRect(item.x, item.y, item.w, item.h);
                            ctx.strokeRect(item.x, item.y, item.w, item.h);

                            ctx.fillStyle = "#8B008B";
                            ctx.font = "bold 16px Arial";
                            ctx.fillText(item.name, item.x, item.y - 8);

                            ctx.fillStyle = "red";
                            ctx.fillRect(item.x + item.w - HANDLE_SIZE, item.y + item.h - HANDLE_SIZE, HANDLE_SIZE, HANDLE_SIZE);

                            drawPins(item);
                        }});
                    }}

                    function drawPins(item) {{
                        let pins = getPinCoords(item);
                        pins.forEach(pin => {{
                            let nearest = getNearestHole(pin.x, pin.y);
                            if (nearest) {{
                                ctx.beginPath();
                                ctx.moveTo(pin.x, pin.y); ctx.lineTo(nearest.x, nearest.y);
                                ctx.strokeStyle = "rgba(0,0,0,0.5)"; ctx.setLineDash([5, 5]); ctx.stroke(); ctx.setLineDash([]);

                                ctx.beginPath();
                                ctx.arc(nearest.x, nearest.y, 7, 0, 2 * Math.PI, false);
                                ctx.strokeStyle = "black"; ctx.lineWidth = 2; ctx.stroke();
                            }}

                            ctx.beginPath();
                            ctx.arc(pin.x, pin.y, 6, 0, 2 * Math.PI, false);
                            ctx.fillStyle = pin.color; ctx.fill(); ctx.strokeStyle = 'white'; ctx.stroke();
                        }});
                    }}

                    function getPinCoords(item) {{
                        let pins = [];
                        if (['fet', 'mosfet'].includes(item.type)) {{
                            let mid_x = item.x + (item.w / 2);
                            pins.push({{name: "Source", x: mid_x, y: item.y, color: "blue"}});
                            pins.push({{name: "Drain", x: mid_x, y: item.y + (item.h / 2), color: "green"}});
                            pins.push({{name: "Gate", x: mid_x, y: item.y + item.h, color: "red"}});
                        }} else if (['res', 'led', 'cap', 'jump'].includes(item.type)) {{
                            if (item.w > item.h) {{
                                let mid_y = item.y + (item.h / 2);
                                pins.push({{name: "Pin1", x: item.x, y: mid_y, color: "red"}});
                                pins.push({{name: "Pin2", x: item.x + item.w, y: mid_y, color: "red"}});
                            }} else {{
                                let mid_x = item.x + (item.w / 2);
                                pins.push({{name: "Pin1", x: mid_x, y: item.y, color: "red"}});
                                pins.push({{name: "Pin2", x: mid_x, y: item.y + item.h, color: "red"}});
                            }}
                        }} else {{
                            pins.push({{name: "Pin1", x: item.x, y: item.y + item.h, color: "red"}});
                            pins.push({{name: "Pin2", x: item.x + item.w, y: item.y + item.h, color: "red"}});
                        }}
                        return pins;
                    }}

                    function getMousePos(evt) {{
                        const rect = canvas.getBoundingClientRect();
                        const scaleX = canvas.width / rect.width; const scaleY = canvas.height / rect.height;
                        return {{ x: (evt.clientX - rect.left) * scaleX, y: (evt.clientY - rect.top) * scaleY }};
                    }}

                    canvas.addEventListener('mousedown', function(e) {{
                        const pos = getMousePos(e);
                        for (let i = items.length - 1; i >= 0; i--) {{
                            let item = items[i];
                            if (pos.x >= item.x + item.w - HANDLE_SIZE && pos.x <= item.x + item.w && pos.y >= item.y + item.h - HANDLE_SIZE && pos.y <= item.y + item.h) {{
                                isResizing = true; dragTarget = item; canvas.style.cursor = 'nwse-resize'; return;
                            }}
                            if (pos.x >= item.x && pos.x <= item.x + item.w && pos.y >= item.y && pos.y <= item.y + item.h) {{
                                isDragging = true; dragTarget = item; offsetX = pos.x - item.x; offsetY = pos.y - item.y; canvas.style.cursor = 'grabbing'; return;
                            }}
                        }}
                    }});

                    canvas.addEventListener('mousemove', function(e) {{
                        const pos = getMousePos(e);
                        if (isDragging && dragTarget) {{
                            dragTarget.x = pos.x - offsetX; dragTarget.y = pos.y - offsetY;
                            draw();
                        }}
                        else if (isResizing && dragTarget) {{
                            let newW = pos.x - dragTarget.x; let newH = pos.y - dragTarget.y;
                            if (newW > 20) dragTarget.w = newW; if (newH > 20) dragTarget.h = newH;
                            draw();
                        }}
                    }});

                    canvas.addEventListener('mouseup', function(e) {{
                        isDragging = false; isResizing = false; dragTarget = null; canvas.style.cursor = 'default';
                        originalItems = JSON.parse(JSON.stringify(items));
                    }});

                    canvas.addEventListener('mouseleave', function(e) {{
                        isDragging = false; isResizing = false; dragTarget = null; canvas.style.cursor = 'default';
                    }});

                    function generateNetlist() {{
                        let netlistMap = {{}};

                        items.forEach(item => {{
                            let pins = getPinCoords(item);
                            pins.forEach(pin => {{
                                let nearestHole = getNearestHole(pin.x, pin.y);
                                if (nearestHole && nearestHole.net_id) {{
                                    if (!netlistMap[nearestHole.net_id]) netlistMap[nearestHole.net_id] = [];
                                    netlistMap[nearestHole.net_id].push(`${{item.name}}(${{pin.name}})`);
                                }}
                            }});
                        }});

                        let out = "🔌 [최종 추출된 전기적 회로 넷리스트]\\n\\n";
                        let validConnections = false;

                        out += "🔗 [상호 연결된 노드]\\n";
                        for (let net in netlistMap) {{
                            if (netlistMap[net].length > 1) {{
                                out += `  - ${{net}} :  ${{netlistMap[net].join("  ↔  ")}}\\n`;
                                validConnections = true;
                            }}
                        }}

                        if (!validConnections) out += "  (아직 연결된 부품이 없습니다. 색칠된 선을 공유하도록 다리를 놔주세요!)\\n";

                        out += "\\n📌 [단일 연결 (혼자 꽂힌 핀)]\\n";
                        for (let net in netlistMap) {{
                            if (netlistMap[net].length === 1) {{
                                out += `  - ${{net}} :  ${{netlistMap[net][0]}}\\n`;
                            }}
                        }}

                        outputConsole.innerText = out;
                    }}
                </script>
                """

                display(HTML(html_code))
